In [ ]:
#librerías y útiles

# Instalación de dependencias (ejecutar una sola vez)
# !pip install missingno fancyimpute

# Librerías estándar
import numpy as np
import pandas as pd

# Configuración de pandas para cambiar el formatod de como salen los floats
#pd.set_option para configuraciones internas.
#display.float_format es el tipo de cambio que se hará
#Luego esta una función anonima para poner cada número
#con solo 2 decimales. (Solo para alteraciones en salidas)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

#ajustar el tamaño de las fuentes, líneas y marcadores de los gráficos para
#que se adapaten al medio donde se van a exponer. 
#.set_context ajusta el tamaño de las fuentes
#"talk" es un preajuste disponible, incrementando el tamaño de todos los elementos.
#hay distintas opciones como "paper" para artículos científicos, 'notebook' para dejarlo
#por defecto y 'poster' para que sea grande para posteres de conferencias. 
sns.set_context('paper')
import plotly.express as px

# Para la visualización de datos faltantes
import missingno as msno

# Imputación avanzada - sklearn
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor

print("✅ Librerías cargadas correctamente")

In [2]:
# Importar librerías
import pandas as pd
import matplotlib.pyplot as plt #gráficos
import seaborn as sns #gráficos
import numpy as np #cálculos numéricos
import os

In [3]:
ruta = "../datos/datos_banco_limpio.csv"
data = pd.read_csv(ruta)

In [4]:
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143.0,yes,no,unknown,5,may,261.0,1,-1.0,0,unknown,no
1,44,technician,single,secondary,no,29.0,yes,no,unknown,5,may,151.0,1,-1.0,0,unknown,no
2,33,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5,may,76.0,1,-1.0,0,unknown,no
3,47,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5,may,92.0,1,-1.0,0,unknown,no
4,33,unknown,single,unknown,no,1.0,no,no,unknown,5,may,198.0,1,-1.0,0,unknown,no


In [5]:
# Forma de hacer un filter de R, pero en python

#data[data['column_name'] == 'value'] #notese que se pone la condición
#dentro de los corchetes 

data[data["age"]<20]

#Forma de hacer el filter con un select
data[data["age"]<20]["age"]

#el select con varias columnas
data[data["age"]<20][["age","job"]]

#Otra forma de hacerlo es con un método query
data.query("age < 20")[["age","job"]]

#otra forma de hacerlo es con un .loc
data.loc[data["age"]<20, ["age","job"]]



,age,job
30780,19,student
31030,19,student
31252,19,student
31293,19,student
31421,19,student
31481,19,student
32158,19,student
33763,19,student
33778,19,student
34270,19,student


A pesar de que todos generan el mismo resultado tiene diferencias en su eficiencia y el como pandas maneja en la ram la vectorización de los datos. 

1. La más óptima `data.loc[data["age"]<20, ["age","job"]]`: pues hace el filtro de filas y seleción de columnas en un solo paso. No crea copias basura temporales en la mem y permite modificar los datos directamente. 

2. La menos optima `data[data["age"]<20][["age","job"]]`: Sufre de indexación encadenada pues obliga a pandas a trabjar en 2 pasos, haciendo el filtro y luego haciendo el select de las columnas. Si el data frame es muy grande va a tomar mucho tiempo. 

3. La intermedia `data.query("age < 20")[["age","job"]]`: también tiene indexación encadenada pero cambia el código a código de máquina en c o c++, por lo que para datasets pequeños es más lenta que .loc, pero para datasets grandes puede ser igual de eficiente. 

En conclusión **evitar a toda costa [][]** y usar cualquiera de las dos entre .query() y .loc(). Pero más recomendable .loc()

In [6]:
#las variables según su tipo de dato
#pueden extraerse como sigue: 

data.select_dtypes(include="number") #variables numéricas
data.select_dtypes(include="object") #variables de tipo objeto

#pero notese que esto retorna un subconjunto del dataframe, es decir, 
#un objeto de tipo data frame pero haciendo un select de las variables
#según su tipo de dato. 

#Para obtener los nombres de las variables es con .columns
#que viene a aser el método equivalente a names() en R.
#Además, lo retorna en un objeto de tipo Index, que es una entidad del objeto dataframe.
#es como una especie de lista. 
data.select_dtypes(include="number") #variables numéricas
type(data.select_dtypes(include="number")) # de tipo dataframe
type(data.select_dtypes(include="number").columns) # de tipo Index
type(data.select_dtypes(include="number").columns.values) # de tipo ndarray que es unv ector de numpy

#hacer type() es la forma equivalente a hacer class() en R. Pues nos dice que tipo de objeto es.
#para saber el tipo de dato de una variable, es decir, si es numérica, categórica, etc. se hace con .dtypes
#pues una cosa es saber el tipo de objeto y otra cosa es saber el tipo de dato de una variable.
data.dtypes 
data["age"].dtypes



dtype('int64')

### Datos tipo object y category.

#### Object:
 Tipo de dato por defectoq ue padnas asigna a culquier columna con texto, string o datos mezclados. Guarda cada palabra literal y de forma independiente. Por ejemplo, si la palabra "SI" esta en 10.000 registros entonces se almacena ese texto 10.000 veces. **Debe usarse para columnas de textoq libre que no se repiten casi nunca, como nombres, correos o mentarios**

#### Category:
Es un tipo de dato especial de pandas diseñado específicamente para variables categóricas que tiene un número limitado de valores únicos que se repiten mucho, por ejemplo el estrato, el género, el mes, etc. Lo que hace es que crea un dicionario interno donde asigna valores numéricos a cada categoría y en lugar de guardar ese texto, guarda la categoría para sr más eficiente. **Se usa para optimizar los dat frames**

Con muchos registros, los tipos de datos pueden generar que la base de datos pese mucho o poco, esto debe organizarse en la limpieza de los datos que es el primer paso de todos. Además, category permite tener un orden lógico y como se puede manipular string con datos de tipo object para las de category será con .cat.

Si se quiere passar una columan de tipo object a de tipo category entocnes simplemente se usa: 

'''
datos[columna] = datos[columna].astype("category)
'''

y al reestructuración o mapeo de los valores dentro de la ram se hace de manera automática. También es importante cambiar el tipo de dato de las variables numéricas pues normalmente usan int64 o double64 lo cual es separar mucho espacio inncesario para número pequeños. 

## Forma de cambiar el tipo de dato según los valores de una variable. 

#### Funciones importantes

**np.iinfo y np.finfo** funciones para conocer los límites técnicos de los tipos de datos. Por tanto, no se puede usar iinfo en un float porque los decimales no rigen las mimas reglas de almacenamiento.

**.nunique()** cuenta cuantos valores distintios hay en una columna por lo que si una columna tiene 1000000 de registros pero solo 3 categorías
es esencial convertirla a una variable de tipo category. Que se crea como un diccionario
y se reduce el tamaño de la columna drásticamente.


Veamos que la mayoría de tipos de datos de nuestra base de datos son objetos, hay float de 64 e int 64, lo cula implica que generan espacíos físicos de 64 bits para cada uno de los registros de esa columna específica y por tanto es espacio innecesario. De ahí, entra la necesidad de ahorrar recursos físicos cambiando el tipo de dato. 

Al cambiar el tipo de dato de 64 bits a 8 bits la computadroa lee los datos originales en int64 y crea una copia temporal de esa columna pero asignandole solo 8 bits (1 byte) a cada fila, se sobreescribe la columna vieja con la nueva versión optimizada y los bloques de memoria de 64 bits que quedaron huerfanos quedan como disponibles. 

Si se esta trabajando con un dataset muy grande y es necesario liberar espacio hay que forzar la limpieza con el garbage collector con: 

`import gc`
`gc.collect()`

que sirve para borra esos reciduos y tener más RAM disponible en la computadora para otras actividades simulatáneas. 


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45189 entries, 0 to 45188
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        45189 non-null  int64  
 1   job        45189 non-null  object 
 2   marital    45189 non-null  object 
 3   education  45189 non-null  object 
 4   default    45189 non-null  object 
 5   balance    45189 non-null  float64
 6   housing    45189 non-null  object 
 7   loan       45189 non-null  object 
 8   contact    45189 non-null  object 
 9   day        45189 non-null  int64  
 10  month      45189 non-null  object 
 11  duration   45189 non-null  float64
 12  campaign   45189 non-null  int64  
 13  pdays      45189 non-null  float64
 14  previous   45189 non-null  int64  
 15  poutcome   45189 non-null  object 
 16  y          45189 non-null  object 
dtypes: float64(3), int64(4), object(10)
memory usage: 5.9+ MB


In [24]:
#Tenemos los número máximo y mínimio de cada tipo de dato
#en este caso para datos de tipo int
np.iinfo(np.int64) #8 bytes por registro
np.iinfo(np.int32) # 4 bytes por registro
np.iinfo(np.int16) # 2 bytes por registro
np.iinfo(np.int8) # 1 byte por registro

#Para datos de tipo float
np.finfo(np.float64) #8 bytes por registro
np.finfo(np.float32) # 4 bytes por registro
np.finfo(np.float16) # 2 bytes por registro


finfo(resolution=0.001, min=-6.55040e+04, max=6.55040e+04, dtype=float16)

#### Recordemos las medidas de unidades de medida en informática para el tamaño de los datos y capacidad de almacenamiento. 

1. bit, representa 1 o 0. 2^0 combinaciones
2. byte, son 8 bits, y son 8 1´s y 0´s. 2^8 combinaciones. 
3. kilobyte, 1024 bytes, que es 2^10 combinaciones. 
4. Megabyte, 1024 kilobytes. 
5. gigabyte, 1024 megabytes. 
6. Terabyte, 1024 gigas. 

In [8]:
#A continuación una función para la reducción de memoria de un dataframe. La función es la siguiente:
##forma de reducir los tamaños de los dataframe es
#revisando los valores máximos y mínimos de las variables cuantitativas
#y con ello poder darles un espacio apto en el tipo de dato para que no
#se le de más espacio del necesario. Esto es una muy buena práctica para
#trabajar con cualquier dataframe ya que puede reducir mucho el espacio
#que ocupan.

def reduccion_variables(df, nombreDf):
    print("Data frame: " + str(nombreDf))
    start_mem = df.memory_usage().sum() / 1024**2 #calculo de los bytes que
    #gasta el data frame y llevarlo a MB
    print(f'Memoria inicial del DataFrame: {start_mem:.2f} MB')

    for col in df.columns: #iniciando por cada una de las variables.
        col_type = df[col].dtype

        # 1. Optimización de Números
        #evalua si la columna no es un objeto(string) o ya es de tipo category
        #notese que para ver que sea de tipo categoría de hace con isinstance.
        if col_type != object and not isinstance(col_type, pd.CategoricalDtype):
          #si no se cumple lo anterior es porque la variable es de tipo númerico
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                # Usamos finfo para floats
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

        # 2. Optimización de Objetos (Strings repetitivos)
        else:
            # Si el texto se repite mucho, 'category' ahorra hasta el 90% de RAM
            num_unique = df[col].nunique()
            if num_unique / len(df) < 0.5: # Si menos del 50% son únicos
                df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memoria final: {end_mem:.2f} MB')
    print(f'Reducción neta: {100 * (start_mem - end_mem) / start_mem:.1f}%')
    print()
    return df

In [9]:
data = reduccion_variables(data, "data")

Data frame: data
Memoria inicial del DataFrame: 5.86 MB
Memoria final: 0.95 MB
Reducción neta: 83.8%



In [10]:
import gc
gc.collect() #libera memoria de variables que ya no se usan 
#y el número que se retorna es el número de objetos que se liberaron de memoria.
# es el conteo directamente. 

20

In [11]:
#Ahora veamos de nuevo la información del dataframe para ver si se redujo el tamaño de memoria que ocupa.
data.info()
#notese que la reducción fue drástica.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45189 entries, 0 to 45188
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   age        45189 non-null  int8    
 1   job        45189 non-null  category
 2   marital    45189 non-null  category
 3   education  45189 non-null  category
 4   default    45189 non-null  category
 5   balance    45189 non-null  float32 
 6   housing    45189 non-null  category
 7   loan       45189 non-null  category
 8   contact    45189 non-null  category
 9   day        45189 non-null  int8    
 10  month      45189 non-null  category
 11  duration   45189 non-null  float16 
 12  campaign   45189 non-null  int8    
 13  pdays      45189 non-null  float16 
 14  previous   45189 non-null  int8    
 15  poutcome   45189 non-null  category
 16  y          45189 non-null  category
dtypes: category(10), float16(2), float32(1), int8(4)
memory usage: 972.9 KB


In [12]:
data.head()

#Notese que directamente los datos de tipo category pasan a verse 
#como texto pues es el mapeo que hace python de manera automática. 
#pero en Ram son números. 

c:\Users\mtang\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143.0,yes,no,unknown,5,may,261.0,1,-1.0,0,unknown,no
1,44,technician,single,secondary,no,29.0,yes,no,unknown,5,may,151.0,1,-1.0,0,unknown,no
2,33,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5,may,76.0,1,-1.0,0,unknown,no
3,47,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5,may,92.0,1,-1.0,0,unknown,no
4,33,unknown,single,unknown,no,1.0,no,no,unknown,5,may,198.0,1,-1.0,0,unknown,no


<span style="color:red;">**IMPORTANTE**</span>
Ya no se puede seguir usando CSV, pues CSV no tiene memoria para guardar los tipos de datos avanzados porque es un archivo de texto plano y eso es directamente una limitación física del CSV.

Por tanto, al correr la función de optimizcióny ver lo cambios con data.info(), todo ocurre dentro de la ram y hacer .to_csv(), pandas de ve obligado a traducir toda la estucutura a texto plano y por defecto se ponen esos tipos de datos que consumen mucho espacio, una solución es la siguiente: 

1. Usar un formato de archivo moderno, dejar de usar CSV sino un formato para Big Data que si guardan los tipo de datos, la optimización y la compresión nativa de la ram, por tanto, hay que instalar las librerias

`pyarrow` y `fastparquet`

2. Otra solución que no es nada recomendada pero posible sería al leer los archivos con el csv, directamente ahcer el tipo de cambio de tipo de datos para cada una de las variables. 

tipos_optimizados = {
    'edad': 'int8',
    'ingresos': 'float32',
    'mes': 'category'
}

Leemos el CSV obligándolo a aplicar la optimización desde el segundo cero
data = pd.read_csv('../datos/datos_banco_limpio_optimizado.csv', dtype=tipos_optimizados)


In [13]:
#guardemos los datos
# Para guardar manteniendo la optimización intacta
data.to_parquet('../datos/datos_banco_limpio_optimizado.parquet')

#### Recordemos como particionar el espacio para poner varios gráficos
Para imprimir varios graficos a la vez hay que usar matplotlib.pyplot con la función subplots de tal manera que se indica comos e particiona el espacio y de que tamaño quedarán los gráficos. 

`fig,axes = plt.subplots(filas,columnas, figsize(ancho, alto))`

y luego en las respectiva gráfica se indica la posición poniendo como argumento el axes[#,#] y ahí se dicen las coordenadas de donde quedará cada gráfico en específico. 

Para ponder títulos o etiquetas se usa también la coordenada y es como si fuera una función. La coordenada se vuelve el objeto del gráfico. 

`axes[#,#].set_title("titulo")
axes[#,#].set_xlabel("etiqueta_eje_x")
`

Para poner una línea en el gráfico
`axes[#,#].axvline(x=lugar_del_eje_x, color='red', linestyle='--', linewidth=2, label=f'etiqueta_linea')
axes[#,#].legend() # Muestra el cuadro con la etiqueta de la línea
`

In [ ]:
import matplotlib.pyplot as plt
import seabonr as sns

taxis = sns.load_dataset('taxis')
fig, axes = plt.subplots(1,2, figsize=(14,5))
sns.histplot(data = taxis,
                x = 'total',
                ax=axes[0],
                kde=True,
                color="skyblue") #con kde = true para poner la densidad. 
axes[0].set_title("Escala original")
axes[0].set_xlabel("Total")


sns.histplot(data = taxis,
                x = 'total',
                ax=axes[1],
                kde=True,
                color="salmon",
                log_scale=True)
axes[1].set_title("Escala logaritmica")
axes[1].set_xlabel("Total en escala logaritmica")

plt.tight_layout() # para ajustar espacios. 

# --- Uso de los ejes correctamente ---
### axis = 0 y axis = 1

La diferencia entre axis=0 y axis=1
Para entenderlo de forma visual, imagina tu tabla de datos (DataFrame) como una cuadrícula con dos ejes:

axis=0 (Filas / Eje Vertical): Si no pones nada, Pandas busca por defecto en las filas. Sirve para borrar registros completos (renglones) basándose en su número de índice; es decir,  VERTICALMENTE SE REFIERE A LA REFERENCIA DE CADA FILA POR SU RESPECTIVO ÍNDICE. 

axis=1 (Columnas / Eje Horizontal): Al poner axis=1, le dices a Pandas: "No busques hacia abajo, busca hacia los lados". Así encuentra la columna con el nombre que le diste para poder borrarla por completo de arriba a abajo; es decir, HORIZONTALMENTE SE REFIERE A BUSCAR EL NOMBRE DE LA COLUMNA. 

Matemáticas	Programación
Juan	10	9
Maria	8	10


Caso 1: Usando axis=0 (Borrar Filas)
Si queremos borrar a Juan de la tabla, tenemos que buscar en el eje de las filas (hacia abajo).

nueva_tabla = notas_df.drop(['Juan'], axis=0)
El resultado sería:     Matemáticas Programación
                    Maria   8           10

Caso 2: Usando axis=1 (Borrar Columnas)
Si lo que queremos es eliminar la materia de Matemáticas por completo, tenemos que buscar en el eje de las columnas (hacia los lados).

nueva_tabla = notas_df.drop(['Matemáticas'], axis=1)
    Programación
Juan	9
Maria	10


Con axis = 0 se toman las columnas como categorías entocnes se tiene en cuenta los valores con respecto al eje vertical y pues con axis = 1 es lo contrario. 


# --- Encoding ---

principalmente se usa en 2 contextos: 
1. **Diccionario de traducción** para el tipo de letras como el encoding utf-8 que es el estandar moderno y más usado que soporta cualquier idioma y emoji.

2. **Encodings categóricos**: Es el proceso de convertir variablers categóricas(texto) en valores numércios. Los algoritmos de machine learning son ecuaciones matemáticas gigantes. No pueden multiplicar o sumar un apalabra por lo que necesitan ser numéricos para poder entrenarse. 

¿Como hacerlo?
Existen diferentes estrategias de encoding según el tipo de dato. 

A) **One-Hot encoding (Para datos sin un orden jerárquico)**.

Lo que hace es crear una nueva columna por cada una de las categorías que habían en la varaible y decir si el registro pertenece a esa categoría o no, parecido a las variables indicadoras. 

Por ejemplo, si tengo una columna que sea tipo de auto y tenga un registro con SEDAN y otro con SUV, entonces se crean dos columnas Auto_Sedan y Auto_SUV y se dice si el registro pertenece al auto sedan o al auto SUV.

con panas: 
`
df_codificado = pd.get_dummies(df, columnas=['variable_categorica'])
`

B) **Label/Ordinal Encoding (Para datos con un orden natural)**.
Lo que hace es asignar un número entero consecutivo a cada categoría, ideal si hay una jerarquía  

(ej. "Bajo", "Medio", "Alto" $\rightarrow$ 0, 1, 2)

con Sickit-Learn:
`
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder() #instancia del encoder

df['Nivel_Codificado'] = encoder.fit_transform(df[['Nivel_Estudios']])
`

## np.where()
`np.where()` es el equivalente a un bloque if-else vectorial en NumPy y Pandas; es decir, un condicional vectorizado.

Sirve para evaluar una condición sobre un conjunto completo de datos (como una columna de un DataFrame o un arreglo de NumPy) y elegir entre dos opciones según el resultado: si se cumple la condición asigna el valor A, y si no se cumple asigna el valor B.

$$\text{np.where}(\underbrace{\text{condición}}_{\text{Evalúa a True/False}}, \ \underbrace{\text{valor\_si\_True}}_{\text{Lo que pone si cumple}}, \ \underbrace{\text{valor\_si\_False}}_{\text{Lo que pone si NO cumple}})$$

In [ ]:
import numpy as np
import pandas as pd

notas = pd.Series([8, 4, 9, 3, 6])

# Si la nota es >= 5 asigna 'Aprobado', de lo contrario 'Reprobado'
resultado = np.where(notas >= 5, 'Aprobado', 'Reprobado')

print(resultado)
# Resultado: ['Aprobado', 'Reprobado', 'Aprobado', 'Reprobado', 'Aprobado']

## función zip()
La función integrada zip() sirve para combinar dos o más secuencias (listas, tuplas, etc.) y recorrerlas en paralelo elemento por elemento dentro de un ciclo for.

Toma el primer elemento de cada secuencia y los agrupa en una tupla, luego el segundo de cada una, y así sucesivamente.


In [1]:
#Ej: Si tienes dos listas independientes de la misma longitud:
nombres = ['Ana', 'Luis', 'Carlos']
edades = [25, 30, 35]

# Sin zip tendrías que usar rangos e índices (menos limpio):
# for i in range(len(nombres)): ...

# Con zip las recorres al mismo tiempo de forma elegante:
for nombre, edad in zip(nombres, edades):
    print(f"{nombre} tiene {edad} años")

Ana tiene 25 años
Luis tiene 30 años
Carlos tiene 35 años


**Un detalle importante sobre la longitud**
Si las listas que pasas a zip() no tienen el mismo tamaño, zip() se detendrá automáticamente cuando la lista más corta se termine (descartando los elementos sobrantes de las más largas).

In [2]:
letras = ['a', 'b', 'c', 'd']
numeros = [1, 2]

for l, n in zip(letras, numeros):
    print(l, n)

# Salida (solo 2 iteraciones):
# a 1
# b 2

a 1
b 2


## .Where() en data frames
Sirve para filtrar o reemplazar valores basados en una condición lógica, manteniendo la estructura y dimensiones originales del data frame. 
Tiene un comportamiento casi opuesto al np.where() de NumPy en cuanto a cómo procesa los argumentos.

¿Cómo funciona df.where()?
La regla fundamental de df.where() es esta:

Donde la condición se cumple (True): Conserva los valores originales.

Donde la condición NO se cumple (False): Reemplaza los valores por otro valor (por defecto usa NaN).

`df.where(condicion, other=np.nan)`

Ejemplo 1: Reemplazar datos que no cumplen una regla con NaN.
Imagina que tienes datos de edades y quieres invalidar las edades menores a 18 (convertirlas a NaN):


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({'edad': [15, 25, 30, 12, 18]})

# Mantiene el valor original si edad >= 18; si no, pone NaN
df_limpio = df.where(df['edad'] >= 18)

print(df_limpio)

# edad
# 0   NaN
# 1  25.0
# 2  30.0
# 3   NaN
# 4  18.0

np.cumsum is a NumPy function that returns the cumulative sum of elements along a given axis.

In [ ]:
import numpy as np

arr = np.array([1, 2, 3, 4])
result = np.cumsum(arr)

print(result)
# Output: [ 1,  3,  6, 10]
# (1, 1+2, 1+2+3, 1+2+3+4)


## pipeline en scikit-learn.
Es una herramienta que permite encadenar múltiples pasos de pre-procesamiento de datos y un modelo final en un solo objeto. En lugar de aplicar transormaciones a los datos paso a paso de foma manual. Pipeline lo que hace es empaquetar todo el flujo de trabajo para que funcione en una sola unidad. 

Ej
```python
pipe = Pipeline([
    ('scaler', StandardScaler()),      # Paso 1: Transformación
    ('model',  LinearRegression()),    # Paso 2: Modelo final
])
```

La lsita dentrod el Pipeline contiene tuplas de la forma: ('nombre_del_paso',objeto):

Paso 1 ('scaler'): Aplica StandardScaler() que es la estandarización llevando a que una varible tenga media cero y varianza 1. 

Paso 2 ('model'): Defnie el algoritmo final, en este caso es una regresión lineal.

Y pues como ambas son instancias, hay que aplicarles el método .fit, por tanto,  para aplicar la estandarización y el algoritmo de regresióni lineal a la vez, se hace con: 

```python
pipe.fit(X_train, y_train)
```

De tal manera que se estandarizán todos los datos  y el entreno con el respectivo algoritmo se hace en un solo paso mucho más rápido. 

Hacerlo de forma manual, sería algo como: 

```python
# ❌ Forma manual (propensa a errores)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # ¡Cuidado con no usar fit aquí!

model = LinearRegression()
model.fit(X_train_scaled, y_train)
```

Notese que en el caso manual, para hacer predicciones y evaluar el modelo usamos directamente la instancia del modelo de regresión(en este caso) ajustado para hacer las predicciones: 

``` python
# 1. Instanciar y entrenar escalador
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 2. Instanciar y entrenar modelo
model = LinearRegression()
model.fit(X_train_scaled, y_train) #Se entrenó el modelo con las variables 
# estandarizadas.

# 3. PREDECIR (¡Paso peligroso si lo olvidas!):
# Tienes que recordar escalar X_test PRIMERO con el scaler de entrenar
X_test_scaled = scaler.transform(X_test) #escalamos
predicciones = model.predict(X_test_scaled)
```

Pero con el **pipeline**, como directamente ya se estandarizaron y se ajustarón esos datos, nuestra nueva instancia a usar será `pipe`:

```python
# 1. Crear el pipeline (agrupa scaler + modelo)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# 2. Entrenar todo el flujo
pipe.fit(X_train, y_train)

# 3. PREDECIR:
# Le pasas X_test directo (en sucio/sin escalar). 
# 'pipe' escala X_test internamente y luego se lo pasa al modelo.
# porque sabe que tiene un trasnformador y se tiene que usar primero
#ya que con eso fue que se ajustó el modelo. 
predicciones = pipe.predict(X_test) 
```


## .fit vs .fit_transform() vs transform() en sickit-learn
La diferencia fundamental está en lo que hacen y en sobre qué datos se deben aplicar.

En Scikit-Learn, los algoritmos de preprocesamiento (como escaladores, codificadores o imputadores) tienen dos tareas distintas: aprender de los datos y transformar los datos.

1. .fit() $\rightarrow$ "Aprender"El método .fit() solo calcula y memoriza los parámetros o estadísticas internas necesarias para la transformación, pero NO modifica los datos.Ejemplo con StandardScaler: .fit(X_train) calcula y guarda la media ($\mu$) y la desviación estándar ($\sigma$) de cada columna de X_train.Resultado: No devuelve datos transformados, solo actualiza el estado interno del objeto (guarda los parámetros aprendidos).

2. .fit_transform() $\rightarrow$ "Aprender + Transformar" (En un solo paso)El método .fit_transform() hace las dos tareas de forma consecutiva y optimizada:Ejecuta .fit() para aprender los parámetros de los datos. Ejecuta .transform() para aplicar la transformación inmediatamente a esos mismos datos.Ejemplo con StandardScaler: .fit_transform(X_train) calcula la media y la desviación estándar de X_train Y ADEMÁS resta la media y divide por la desviación a ese mismo X_train.Resultado: Devuelve una matriz con los datos ya transformados/escalados.

Método	¿Qué hace?	¿Cuándo se usa?
.fit(): Calcula y memoriza parámetros (media, min, max, clases, etc.).	Rara vez se usa solo en preprocesamiento; suele combinarse o usarse en el modelo final.

.transform(): Aplica la transformación usando parámetros previamente memorizados.	Se usa en X_test y en cualquier dato nuevo en producción. Es decir, los datos se transforman (en este caso del método de estandarización) con la media y varianza del conjunto de entrenamiento, para hacer el ejercicio como si fueran casos reales que no se conoce ni su media ni su desviación para poder usar el método. 

.fit_transform(): Aprende los parámetros y transforma los datos al mismo tiempo. Se usa exclusivamente en X_train.

In [ ]:
#                ┌───────────────────────┐
#                │     DATOS DE ENTRADA  │
#                └───────────┬───────────┘
#                            │
#              ┌─────────────┴─────────────┐
#              ▼                           ▼
#       Conjunto TRAIN              Conjunto TEST / Datos nuevos
#    (X_train, y_train)                  (X_test)
#              │                           │
#              ▼                           ▼
#    usar .fit_transform()              usar SOLO .transform()

con `.fit_transform()` que aplicamos al conjunto de entrenamiento es ajustar y transformar esos datos para el modelo. Luego, la instancia del modelo se usa solo `.fit` para calcular y memorizar las ecuaciones y coeficientes que necesita el algoritmo. Por ejemplo: 
* para el caso de una regresión lieneal .fit() calcula y guarda los valores de los coeficientes de regreseión que se hallan con mínimos cuardrados ordinarios, lo que hace el método es llenar internamente atributos con esta información, para este caso los atributos `.coef_` y `.intercept_` de la instancia model

luego, `model.predict(X_test_scaled)` es ajustar los datos de prueba con el modelo ajustado, como siempre. 

y pues para crear el X_teste_scales lo que usamos es directamente la instancia de la estandarización `scaler.transorm(X_test)` para que se estandarize con los datos aprendidos del conjunto de prueba como si fueran datos completamente nuevos pues si se le pone el .fit_transform() pues direcamente se esacala con su respectiva media y desviación y es como si en la vida real conocieras la media y desviación de los datos y pues eso no pasa lo cual es un gran y terrbile error. 

Por tanto, para no quedarse calvo y estresarse por eso (aunque la verdad es bastante sencillo), simplemente hay que usar un `pipe` que solo llama el `.fit()` y a `.predict()` pues esto ya sabe que aplicar y como aplicarlo por lo que ahorra unas cuantas líneas de código. 

In [ ]:
# 1. Cuando ejecutas: pipe.fit(X_train, y_train)
#    │
#    ├──► Paso 1: 'scaler' (Transformador)
#    │    Ejecuta: scaler.fit_transform(X_train)
#    │    (Calcula la media/desviación de X_train Y lo escala)
#    │
#    └──► Paso 2: 'model' (Estimador Final)
#         Ejecuta: model.fit(X_train_escalado, y_train)
#         (Calcula y guarda los coeficientes del modelo)


# 2. Cuando ejecutas: pipe.predict(X_test)
#    │
#    ├──► Paso 1: 'scaler' (Transformador)
#    │    Ejecuta: scaler.transform(X_test)  <── ¡SOLO .transform()!
#    │    (Usa la media/desviación que memorizó de X_train)
#    │
#    └──► Paso 2: 'model' (Estimador Final)
#         Ejecuta: model.predict(X_test_escalado)
#         (Aplica las betas aprendidas para dar la predicción)

Por tanto, .fit(), que es el `ajustar` es guardar los coeficientes que se necesitan para hacer las predicciones y ya. 

np.meshgrid toma vectores unidimensionales y construye matrices de coordenadas. Es muy útil para evaluar una funcióini sobre toda una rejilla de puntos

In [ ]:
import numpy as np

x = np.array([1, 2, 3])
y = np.array([4, 5])

X, Y = np.meshgrid(x, y)

# X: repite x a lo largo de las filas
# [[1, 2, 3],
#  [1, 2, 3]]

# Y: repite y a lo largo de las columnas
# [[4, 4, 4],
#  [5, 5, 5]]


In [ ]:
# Evalúa z en los 6 puntos de la rejilla al mismo tiempo
Z = X**2 + Y**2

Con esas tres matrices (X, Y, Z), funciones como plt.plot_surface() o plt.contour() pueden trazar la figura en 3D o las curvas de nivel.

----------- PRUEBAS -----------------

In [4]:
import numpy as np

In [7]:
beta1_range = np.linspace(0, 4, 100)
beta2_range = np.linspace(-1, 3, 100)
B1, B2 = np.meshgrid(beta1_range, beta2_range)

In [8]:
B1

array([[0.        , 0.04040404, 0.08080808, ..., 3.91919192, 3.95959596,
        4.        ],
       [0.        , 0.04040404, 0.08080808, ..., 3.91919192, 3.95959596,
        4.        ],
       [0.        , 0.04040404, 0.08080808, ..., 3.91919192, 3.95959596,
        4.        ],
       ...,
       [0.        , 0.04040404, 0.08080808, ..., 3.91919192, 3.95959596,
        4.        ],
       [0.        , 0.04040404, 0.08080808, ..., 3.91919192, 3.95959596,
        4.        ],
       [0.        , 0.04040404, 0.08080808, ..., 3.91919192, 3.95959596,
        4.        ]], shape=(100, 100))

In [ ]:

B1, B2 = np.meshgrid(beta1_range, beta2_range)

# 3. Calcular la suma de cuadrados de los residuos para cada combinación de (beta_1, beta_2)
SSE = np.zeros(B1.shape)
for i in range(B1.shape[0]):
    for j in range(B1.shape[1]):
        #crear a Z.
        y_pred = B1[i, j] * X1 + B2[i, j] * X2
        #crear la suma de cuadrados de los residuos
        SSE[i, j] = np.sum((y - y_pred)**2)
